In [1]:
import numpy as np
import tensorflow as tf
from tensorflow import keras

In [2]:
# Source sentences (English) and Target sentences (French)
raw_data = [
    ("go", "va"),
    ("run", "cours"),
    ("hi", "salut"),
    ("stop", "arrete"),
    ("wait", "attends"),
    ("see you", "a bientot"),
    ("i try", "j essaye"),
    ("i won", "j ai gagne")
]

In [3]:
# Add start (<start>) and end (<end>) tokens to target sentences
source_text = [pair[0] for pair in raw_data]
target_text_in = [f"<start> {pair[1]} "for pair in raw_data]
target_text_out = [f"{pair[1]} <end> "for pair in raw_data]

In [4]:
# Vectorize English (Source)
src_vector = tf.keras.layers.TextVectorization(output_mode='int', output_sequence_length=4)
src_vector.adapt(source_text)

In [5]:
# Vectorize French (Target)
text_vector = tf.keras.layers.TextVectorization(output_mode='int', output_sequence_length=5)
text_vector.adapt(target_text_in + target_text_out)

In [6]:
source_vocab_size = len(src_vector.get_vocabulary())
text_vocab_size = len(text_vector.get_vocabulary())

In [7]:
# Prepare numeric arrays
enc_input_data = src_vector(np.array(source_text)).numpy()
dec_input_data = text_vector(np.array(target_text_in)).numpy()
dec_target_data = text_vector(np.array(target_text_out)).numpy()

In [8]:
# 2. Model Architecture
latent_dim =32

# --- Encoder ---
encoder_inputs = tf.keras.layers.Input(shape=(4,), name="encoder_inputs")
enc_emb = tf.keras.layers.Embedding(source_vocab_size, 16, mask_zero=True)(encoder_inputs)
encoder_lstm = tf.keras.layers.LSTM(latent_dim, return_state=True, name="encoder_lstm")
_, state_h, state_c = encoder_lstm(enc_emb)
encoder_states = [state_h, state_c]

In [10]:
# --- DECODER ---
decoder_inputs = tf.keras.layers.Input(shape=(5,),name="decoder_inputs")
dec_emb_layer = tf.keras.layers.Embedding(text_vocab_size, 16 ,mask_zero=True)
dec_emb = dec_emb_layer(decoder_inputs)
dec_lstm = tf.keras.layers.LSTM(latent_dim, return_sequences=True, return_state=True, name = "decoder_lstm")

decoder_outputs, _, _ = dec_lstm(dec_emb, initial_state=encoder_states)

decoder_dense = tf.keras.layers.Dense(text_vocab_size, activation="softmax", name="decoder_dense")
decoder_outputs = decoder_dense(decoder_outputs)

In [11]:
# Complete Trainig Model

training_model = tf.keras.Model([encoder_inputs, decoder_inputs], decoder_outputs)

training_model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ encoder_inputs (InputLayer)   │ (None, 4)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_inputs (InputLayer)   │ (None, 5)                 │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding (Embedding)         │ (None, 4, 16)             │             192 │ encoder_inputs[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ not_equal (NotEqual)          │ (None, 4)                 │               0 │ encoder_inputs[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ embedding_2 (Embedding)       │ (None, 5, 16)             │             240 │ decoder_inputs[0][0]       │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ encoder_lstm (LSTM)           │ [(None, 32), (None, 32),  │           6,272 │ embedding[0][0],           │
│                               │ (None, 32)]               │                 │ not_equal[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_lstm (LSTM)           │ [(None, 5, 32), (None,    │           6,272 │ embedding_2[0][0],         │
│                               │ 32), (None, 32)]          │                 │ encoder_lstm[0][1],        │
│                               │                           │                 │ encoder_lstm[0][2]         │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ decoder_dense (Dense)         │ (None, 5, 15)             │             495 │ decoder_lstm[0][0]         │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 13,471 (52.62 KB)

 Trainable params: 13,471 (52.62 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
training_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
    loss="sparse_categorical_crossentropy",
    metrics=['accuracy']
)

In [13]:
history = training_model.fit(
    [enc_input_data,dec_input_data],
    dec_target_data,
    epochs=120,
    verbose=1
)
print("Seq2Seq training complete!")

Epoch 1/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 4s 4s/step - accuracy: 0.0000e+00 - loss: 2.7109
Epoch 2/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 76ms/step - accuracy: 0.4000 - loss: 2.6805
Epoch 3/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.4000 - loss: 2.6471
Epoch 4/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 56ms/step - accuracy: 0.4000 - loss: 2.6034
Epoch 5/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step - accuracy: 0.4000 - loss: 2.5442
Epoch 6/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.4000 - loss: 2.4632
Epoch 7/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 45ms/step - accuracy: 0.4000 - loss: 2.3551
Epoch 8/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 72ms/step - accuracy: 0.4000 - loss: 2.2229
Epoch 9/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 74ms/step - accuracy: 0.4000 - loss: 2.0983
Epoch 10/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step - accuracy: 0.4000 - loss: 2.0411
Epoch 11/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step - accuracy: 0.4000 - loss: 2.0409
Epoch 12/120
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 75ms/step - accuracy: 0.4000 - 

In [15]:
# 1. Encoder Model: Takes English, outputs [state_h, state_c]
encoder_model = tf.keras.Model(encoder_inputs, encoder_states)

# 2. Decoder Model: Takes 1 target token + [state_h, state_c], outputs next word probs + new [state_h, state_c]
decoder_state_input_h = tf.keras.layers.Input(shape=(latent_dim,))
decoder_state_input_c = tf.keras.layers.Input(shape=(latent_dim,))
decoder_states_inputs = [decoder_state_input_h, decoder_state_input_c]

# Single-step input for the decoder (1 token at a time)
decoder_single_input = tf.keras.layers.Input(shape=(1,))
dec_emb_inf = dec_emb_layer(decoder_single_input)

# Reuse the trained decoder LSTM and Dense layers
decoder_outputs_inf, state_h_inf, state_c_inf = dec_lstm(
    dec_emb_inf, 
    initial_state=decoder_states_inputs
)
decoder_states_inf = [state_h_inf, state_c_inf]
decoder_outputs_inf = decoder_dense(decoder_outputs_inf)

decoder_model = tf.keras.Model(
    [decoder_single_input] + decoder_states_inputs,
    [decoder_outputs_inf] + decoder_states_inf
)

In [26]:
# Create index-to-word and word-to-index lookup dictionaries
tgt_vocab = text_vector.get_vocabulary()
tgt_word_to_idx = {word: i for i, word in enumerate(tgt_vocab)}
tgt_idx_to_word = {i: word for i, word in enumerate(tgt_vocab)}

start_token_idx = tgt_word_to_idx["start"]
end_token_idx = tgt_word_to_idx["end"]

def translate(input_sentence):
    # Vectorize input English sentence
    input_seq = src_vector(np.array([input_sentence])).numpy()
    
    # Get initial states from Encoder
    states_value = encoder_model.predict(input_seq, verbose=0)
    
    # Start target sequence with '<start>' token (shape: [1, 1])
    target_seq = np.array([[start_token_idx]])
    
    decoded_sentence = []
    stop_condition = False
    
    while not stop_condition:
        # Predict next word distribution and updated states
        output_tokens, h, c = decoder_model.predict(
            [target_seq] + states_value, 
            verbose=0
        )
        
        # Pick the token with highest probability (argmax)
        sampled_token_index = np.argmax(output_tokens[0, -1, :])
        sampled_word = tgt_idx_to_word[sampled_token_index]
        
        # Stop if we hit '<end>' or max length of 6 words
        if sampled_word == "end" or len(decoded_sentence) > 6:
            stop_condition = True
        else:
            decoded_sentence.append(sampled_word)
            
        # Update target sequence to be the token just predicted
        target_seq = np.array([[sampled_token_index]])
        
        # Update states for the next step
        states_value = [h, c]
        
    return " ".join(decoded_sentence)

In [27]:
test_phrases = ["go", "run", "hi", "stop", "see you", "i won"]

for phrase in test_phrases:
    translation = translate(phrase)
    print(f"EN: '{phrase}'  -->  FR: '{translation}'")

EN: 'go'  -->  FR: 'va'
EN: 'run'  -->  FR: 'cours'
EN: 'hi'  -->  FR: 'salut'
EN: 'stop'  -->  FR: 'arrete'
EN: 'see you'  -->  FR: 'a bientot'
EN: 'i won'  -->  FR: 'j ai gagne'
